In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv("../data/telco_customer_churn.csv")

# Look at the first 5 rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
# Check if there are blank spaces in TotalCharges
blank_charges = df[df['TotalCharges'] == " "]
print(f"Number of rows with blank TotalCharges: {len(blank_charges)}")
blank_charges[['tenure', 'MonthlyCharges', 'TotalCharges']].head()

Number of rows with blank TotalCharges: 11


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,
753,0,20.25,
936,0,80.85,
1082,0,25.75,
1340,0,56.05,


**What's happening here:** When a customer is brand new (`tenure == 0`), they haven't been billed yet, so the database recorded their `TotalCharges` as a literal empty space string (`" "`). Pandas sees those spaces and assumes the whole column must be text!

In [5]:
df['Churn'].value_counts(normalize=True) * 100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

This is an **imbalanced dataset**. If we built a lazy model that just guessed "No" for every single person, it would be 73.4% accurate, but it would completely fail to catch a single person who is actually about to cancel their service. This tells us that later on, we shouldn't rely on "Accuracy" to score our model, we will need to look at **Recall** and **F1-Score**.

Now that we know why `TotalCharges` is broken, let's fix it right there in the notebook and look at how features actually relate to a customer churning.

In [6]:
# Replace empty spaces with NaN
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)

# Convert the column to float numeric values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Check for missing values
print("Missing Values per column:")
print(df.isnull().sum())

Missing Values per column:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [7]:
# Cross-tabulate Contract type against Churn
pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100

Churn,No,Yes
Contract,,
Month-to-month,57.290323,42.709677
One year,88.730482,11.269518
Two year,97.168142,2.831858


In [8]:
# Get the average tenure (in months) for those who churned vs those who stayed
df.groupby('Churn')['tenure'].mean()

Churn
No     37.569965
Yes    17.979133
Name: tenure, dtype: float64

The data shows a massive pattern: **Month-to-month contracts are highly volatile** (often seeing over 40% churn), while two-year contracts have almost negligible churn (usually under 3%). Furthermore, the average tenure for people who churn is drastically lower (around 18 months) compared to those who stay (around 38 months).

This means features like `Contract` and `tenure` are going to be incredibly strong predictors for our machine learning model.